# Use merge schema for additive changes

In [6]:
print(spark.conf.get("spark.databricks.delta.schema.autoMerge.enabled"))

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 9, Finished, Available, Finished, False)

false


In [7]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

# Define consistent schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", FloatType(), True)
])

data = [(1, "Alice", 5000.0), (2, "Bob", 6000.0)]
df = spark.createDataFrame(data, schema)

delta_table_path="Tables/dbo/employees"

# Overwrite table
df.write.format("delta") \
    .mode("overwrite") \
    .save(delta_table_path)

# Read table
df = spark.sql("SELECT * FROM employees LIMIT 1000")
display(df)


StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b2eb22a8-b7d9-480d-8574-118f60afc9dd)

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Wrong type for salary (String) and extra column age
schema_wrong = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),    
    StructField("salary", FloatType(), True), 
    StructField("age", IntegerType(), True)   # extra Column
])

data_wrong = [(3, "Charlie", 8000.50, 30)]

df_invalid = spark.createDataFrame(data_wrong, schema_wrong)

# This will fail due to schema enforcement
df_invalid.write.format("delta") \
    .mode("append") \
    .save(delta_table_path)


StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 11, Finished, Available, Finished, False)

AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: b9429f85-98bd-438a-a0bf-11015130b495).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: float (nullable = true)


Data schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: float (nullable = true)
 |-- age: integer (nullable = true)



In [9]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

# Correct salary type and new age column
schema_new = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", FloatType(), True),
    StructField("age", IntegerType(), True)
])

data_new = [(4, "David", 5500.0, 28)]
df_new = spark.createDataFrame(data_new, schema_new)

df_new.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .save(delta_table_path)

df = spark.sql("SELECT * FROM employees LIMIT 1000")
display(df)


StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 771eb1dd-b71e-4787-b789-f922f5881cbb)

In [10]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

# Correct salary type and new age column
schema_new = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", FloatType(), True),
    StructField("age", IntegerType(), True),
    StructField("comm", IntegerType(), True)
])

data_new = [(4, "David", 5500.0, 28,500)]
df_new = spark.createDataFrame(data_new, schema_new)

df_new.write.format("delta") \
    .mode("append") \
    .save(delta_table_path)

df = spark.sql("SELECT * FROM employees LIMIT 1000")
display(df)


StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 13, Finished, Available, Finished, False)

AnalysisException: [DELTA_METADATA_MISMATCH] A metadata mismatch was detected when writing to the Delta table.
- A schema mismatch detected when writing to the Delta table (Table ID: b9429f85-98bd-438a-a0bf-11015130b495).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set: '.option("mergeSchema", "true")'.
For other operations, set the session configuration spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation specific to the operation for details.

Table schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: float (nullable = true)
 |-- age: integer (nullable = true)


Data schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: float (nullable = true)
 |-- age: integer (nullable = true)
 |-- comm: integer (nullable = true)



# Session Level autoMerge

In [12]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled","True")

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 15, Finished, Available, Finished, False)

In [13]:
print(spark.conf.get("spark.databricks.delta.schema.autoMerge.enabled"))

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 16, Finished, Available, Finished, False)

True


In [14]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

# Correct salary type and new age column
schema_new = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("salary", FloatType(), True),
    StructField("age", IntegerType(), True),
    StructField("comm", IntegerType(), True)
])

data_new = [(4, "David", 5500.0, 28,500)]
df_new = spark.createDataFrame(data_new, schema_new)

df_new.write.format("delta") \
    .mode("append") \
    .save(delta_table_path)

df = spark.sql("SELECT * FROM employees LIMIT 1000")
display(df)


StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d6f5c9a4-0f0a-42a0-937d-487d445ee97f)

# Use column mapping for rename and drop operations

# rename column

In [16]:
%%sql
ALTER TABLE employees
RENAME COLUMN id TO empid

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 19, Finished, Available, Finished, False)

Error: [DELTA_UNSUPPORTED_RENAME_COLUMN] Column rename is not supported for your Delta table. 
Please enable Column Mapping on your Delta table with mapping mode 'name'.
You can use one of the following commands.

ALTER TABLE table_name SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

Note, if your table is not on the required protocol version it will be upgraded.
Column mapping requires at least protocol (2, 5)


In [18]:
%%sql
ALTER TABLE employees SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [19]:
%%sql
ALTER TABLE employees
RENAME COLUMN id TO empid

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 22, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

# drop column

In [20]:
%%sql
ALTER TABLE employees
DROP COLUMN comm

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 23, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

# Use type widening for compatible type changes

In [25]:
%%sql
ALTER TABLE employees
ALTER COLUMN empid TYPE LONG

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 28, Finished, Available, Finished, False)

Error: [DELTA_UNSUPPORTED_ALTER_TABLE_CHANGE_COL_OP] ALTER TABLE CHANGE COLUMN is not supported for changing column empid from INT to BIGINT

In [26]:
%%sql
ALTER TABLE employees
SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 29, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [28]:
%%sql
ALTER TABLE employees
ALTER COLUMN salary TYPE double

StatementMeta(, 24731b3a-2faa-4b1b-82c3-7f22e737b8a3, 31, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>